In [ ]:

import json
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torchvision import models

import pandas as pd
from tqdm import tqdm
from torchvision import transforms

In [17]:
relation_class = ["above",	"across from",
"adjacent to",	"against",
"along",	"around",
"at",	"attached to",
"before",	"behind",
"below",	"beneath",
"beside",	"between",
"by",	"carrying",
"connected to",	"contains",
"covered by",	"covering",
"crossing",	"cutting",
"driving",	"eating",
"facing",	"floating on",
"hanging on",	"has",
"has on",	"holding",
"in",	"in front of",
"inside",	"leaning on",
"left of",	"looking at",
"lying on",	"moving on",
"near",	"next to",
"on",	"on top of",
"outside",	"over",
"parked next to",	"parked on",
"playing",	"pulling",
"riding",	"right of",
"says",	"sitting near",
"sitting next to",	"sitting on",
"standing near",	"standing next to",
"standing on",	"stopped on",
"swinging",	"taller than",
"tied to",	"touching",
"under",	"using",
"walking on",	"watching",
"wearing",	"with"]

df = pd.DataFrame(relation_class, columns=['relation_name'])

In [18]:
df.to_csv('relation_classes.csv', index=False)

In [6]:
dataset_path = '../../sg_dataset'

test_graph_path = dataset_path + '/sg_test_annotations.json'
train_graph_path = dataset_path + '/sg_train_annotations.json'
test_images_path = dataset_path + '/sg_test_images/'
train_images_path = dataset_path + '/sg_train_images/'

In [7]:
with open(train_graph_path, 'r') as f:
    train_graphs = json.load(f)

N = len(train_graphs)

In [ ]:
train_graphs[0]['objects']['names']

[{'attributes': [{'text': ['car', 'is', 'gray'], 'attribute': 'gray'},
   {'text': ['car', 'is', 'moving'], 'attribute': 'moving'},
   {'text': ['car', 'is', 'shiny'], 'attribute': 'shiny'}],
  'names': ['car'],
  'bbox': {'y': 346.0, 'x': 541.0, 'w': 319.0, 'h': 128.0}},
 {'attributes': [{'text': ['front wheels', 'is', 'round'],
    'attribute': 'round'},
   {'text': ['front wheels', 'is', 'rubber'], 'attribute': 'rubber'},
   {'text': ['front wheels', 'is', 'black'], 'attribute': 'black'}],
  'names': ['front wheels'],
  'bbox': {'y': 415.0, 'x': 744.0, 'w': 60.0, 'h': 55.0}},
 {'attributes': [{'text': ['back wheels', 'is', 'round'],
    'attribute': 'round'},
   {'text': ['back wheels', 'is', 'rubber'], 'attribute': 'rubber'},
   {'text': ['back wheels', 'is', 'black'], 'attribute': 'black'}],
  'names': ['back wheels'],
  'bbox': {'y': 410.0, 'x': 567.0, 'w': 49.0, 'h': 48.0}},
 {'attributes': [{'text': ['sky', 'is', 'cloudy'], 'attribute': 'cloudy'}],
  'names': ['sky'],
  'bbox':

In [ ]:
""" def calculate_iou(boxA, boxB):
    # box format: [x0, y0, x1, y1]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    
    iou = interArea / float(boxAArea + boxBArea - interArea + 1e-6)
    return iou

# Example Processing Loop
training_data = []

for i in tqdm(range(N), desc="Processing Images"):
    image_filename = test_graphs[i]['filename']
    gt_objects = test_graphs[i]['objects']
    
    image_path = test_images_path + image_filename
    proposals = get_bboxes_from_cpp(image_path)
    
    for prop in proposals:
        x0, y0, x1, y1 = prop
        best_iou = 0
        assigned_label = None
        assigned_attributes = []
        best_gt = None # Track the best GT object for labeling

        for gt in gt_objects:
            gt_box = [
                gt['bbox']['x'], 
                gt['bbox']['y'], 
                gt['bbox']['x'] + gt['bbox']['w'], 
                gt['bbox']['y'] + gt['bbox']['h']
            ]
            
            iou = calculate_iou(prop, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt = gt

        # CASE 1: Positive match (Object + Attributes)
        if best_iou >= 0.5:
            training_data.append({
                'image': image_filename,
                'box': prop,
                'class': best_gt['names'], 
                'attrs': [attr['attribute'] for attr in best_gt['attributes']],
                'is_background': False
            })

        # CASE 2: Background match (Important for training R-CNN)
        elif best_iou < 0.3:
            training_data.append({
                'image': image_filename,
                'box': prop,
                'class': 'background', 
                'attrs': [],
                'is_background': True
            }) """

Processing Images: 100%|██████████| 1000/1000 [17:39<00:00,  1.06s/it]


In [ ]:
rcnn_training_data = pd.read_csv("rcnn_training_data.csv")

attribute_classes = pd.read_csv("attribute_classes.csv")
object_classes = pd.read_csv("object_classes.csv")

In [ ]:
rcnn_training_data.head()

In [ ]:
class RCNNDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform or transforms.Compose([
            transforms.Resize((227, 227)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = f"{self.img_dir}/{row['image']}"
        image = Image.open(img_path).convert('RGB')
        
        # Crop the bounding box [x0, y0, x1, y1]
        bbox = eval(row['box']) 
        cropped_img = image.crop((bbox[0], bbox[1], bbox[2], bbox[3]))
        
        if self.transform:
            cropped_img = self.transform(cropped_img)
            
        # Labels: Object class index and multi-hot attributes
        obj_label = row['class_idx'] 
        attr_labels = torch.tensor(eval(row['attr_vec']), dtype=torch.float32)
        
        return cropped_img, obj_label, attr_labels

In [ ]:
class MultiHeadRCNN(nn.Module):
    def __init__(self, num_objects, num_attributes):
        super(MultiHeadRCNN, self).__init__()
        # Backbone (The paper references Caffe/AlexNet style)
        self.backbone = models.alexnet(pretrained=True)
        
        # The paper extracts 4,096-D features from the fc7 layer 
        feature_dim = self.backbone.classifier[6].in_features
        self.backbone.classifier = nn.Sequential(*list(self.backbone.classifier.children())[:-1])
        
        # Head 1: Object Classification (Softmax-ready)
        self.obj_head = nn.Linear(feature_dim, num_objects)
        
        # Head 2: Attribute Classification (Multi-label Sigmoid-ready)
        self.attr_head = nn.Linear(feature_dim, num_attributes)

    def forward(self, x):
        features = self.backbone(x) # These are your 4,096-D vectors 
        obj_logits = self.obj_head(features)
        attr_logits = self.attr_head(features)
        return obj_logits, attr_logits

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS device")
else:
    device = torch.device("cpu")
    print("MPS not available, using CPU")

train_dataset = RCNNDataset(
    csv_file='rcnn_training_data.csv', 
    img_dir=train_images_path
)

# Define the DataLoader
# batch_size=128 is a standard choice for R-CNN training 
dataloader = DataLoader(
    train_dataset, 
    batch_size=128, 
    shuffle=True, # Critical for mixing background and object samples
    num_workers=4  # Speed up data loading by using multiple CPU cores
)

model = MultiHeadRCNN(num_objects=267, num_attributes=145).to(device)

# Loss functions
criterion_obj = nn.CrossEntropyLoss() # For objects
criterion_attr = nn.BCEWithLogitsLoss() # For multi-label attributes [cite: 285]
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

for epoch in range(10):
    model.train()
    for images, obj_labels, attr_labels in dataloader:
        images, obj_labels, attr_labels = images.to(device), obj_labels.to(device), attr_labels.to(device)
        
        optimizer.zero_grad()
        obj_pred, attr_pred = model(images)
        
        loss_obj = criterion_obj(obj_pred, obj_labels)
        loss_attr = criterion_attr(attr_pred, attr_labels)
        
        total_loss = loss_obj + loss_attr
        total_loss.backward()
        optimizer.step()